In [1]:
import pandas as pd

In [2]:
# 1️⃣ Load raw CSV
df = pd.read_csv("data\incident_event_log.csv")

In [3]:
# 2️⃣ Quick inspection
print(df.head())
print("Shape:", df.shape)
print("Missing Values:", df.isnull().sum())
print("Duplicates:", df["number"].duplicated().sum())
print("Unique incidents:", df["number"].nunique())


       number incident_state  active  reassignment_count  reopen_count  \
0  INC0000045            New    True                   0             0   
1  INC0000045       Resolved    True                   0             0   
2  INC0000045       Resolved    True                   0             0   
3  INC0000045         Closed   False                   0             0   
4  INC0000047            New    True                   0             0   

   sys_mod_count  made_sla    caller_id       opened_by        opened_at  ...  \
0              0      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
1              2      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
2              3      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
3              4      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
4              0      True  Caller 2403  Opened by  397  29/2/2016 04:40  ...   

  u_priority_confirmation         notify problem_id rfc vendor cause

In [4]:
# 3️⃣ Convert date columns
date_cols = ["opened_at", "resolved_at", "closed_at"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

In [5]:
# 4️⃣ Remove unnecessary columns early
columns_to_drop = ["problem_id", "rfc", "vendor", "caused_by", "closed_code",
                   "sys_created_by", "sys_created_at", "sys_updated_by", "sys_updated_at",
                   "contact_type", "location", "knowledge", "u_priority_confirmation", "notify", "cmdb_ci"]
df = df.drop(columns=columns_to_drop)

In [6]:
# 5️⃣ Sort by sys_mod_count (last update)
df_sorted = df.sort_values(["number", "sys_mod_count"])

In [7]:
# 6️⃣ Keep last record per incident (final state)
incident_final = df_sorted.groupby("number").last().reset_index()
print("Final Incident Table Shape:", incident_final.shape)


Final Incident Table Shape: (24918, 21)


In [8]:
# 7️⃣ Feature engineering: resolution hours
incident_final["resolution_hours"] = (
    (incident_final["resolved_at"] - incident_final["opened_at"])
    .dt.total_seconds() / 3600
)

In [9]:
# 8️⃣ Quick stats
print(incident_final[["opened_at", "resolved_at", "resolution_hours"]].head())
print(incident_final["resolution_hours"].describe())
print(incident_final["made_sla"].value_counts())
print(incident_final.groupby("made_sla")["resolution_hours"].mean())

            opened_at         resolved_at  resolution_hours
0 2016-02-29 01:16:00 2016-02-29 11:29:00         10.216667
1 2016-02-29 04:40:00 2016-03-01 09:52:00         29.200000
2 2016-02-29 06:10:00 2016-03-01 02:55:00         20.750000
3 2016-02-29 06:38:00 2016-03-02 12:06:00         53.466667
4 2016-02-29 06:58:00 2016-02-29 15:51:00          8.883333
count    23362.000000
mean       178.171582
std        532.787772
min          0.000000
25%          0.416667
50%         22.100000
75%        148.479167
max       8070.166667
Name: resolution_hours, dtype: float64
True     15803
False     9115
Name: made_sla, dtype: int64
made_sla
False    438.311174
True      17.206605
Name: resolution_hours, dtype: float64


In [10]:
# 9️⃣ Final preview
incident_final.head()

,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,u_symptom,impact,urgency,priority,assignment_group,assigned_to,resolved_by,resolved_at,closed_at,resolution_hours
0,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,...,Symptom 72,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,Resolved by 149,2016-02-29 11:29:00,2016-03-05 12:00:00,10.216667
1,INC0000047,Closed,False,1,0,8,True,Caller 2403,Opened by 397,2016-02-29 04:40:00,...,Symptom 471,2 - Medium,2 - Medium,3 - Moderate,Group 24,Resolver 89,Resolved by 81,2016-03-01 09:52:00,2016-03-06 10:00:00,29.200000
2,INC0000057,Closed,False,0,0,6,True,Caller 4416,Opened by 8,2016-02-29 06:10:00,...,Symptom 471,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 6,Resolved by 5,2016-03-01 02:55:00,2016-03-06 03:00:00,20.750000
3,INC0000060,Closed,False,0,0,3,True,Caller 4491,Opened by 180,2016-02-29 06:38:00,...,Symptom 450,2 - Medium,2 - Medium,3 - Moderate,Group 25,Resolver 125,Resolved by 113,2016-03-02 12:06:00,2016-03-07 13:00:00,53.466667
4,INC0000062,Closed,False,1,0,7,False,Caller 3765,Opened by 180,2016-02-29 06:58:00,...,Symptom 232,1 - High,2 - Medium,2 - High,Group 23,?,Resolved by 62,2016-02-29 15:51:00,2016-03-05 16:00:00,8.883333


In [11]:
incident_final["is_resolved"] = incident_final["resolved_at"].notnull()

In [12]:
#cap = incident_final["resolution_hours"].quantile(0.95)
#incident_final["resolution_hours_capped"] = incident_final["resolution_hours"].clip(upper=cap)

In [13]:
incident_final.to_csv("outputs\incident_cleaned.csv", index=False)